# 🏘️ Auditoría de Valor: ¿Qué hace que una casa sea cara en California?

Tras optimizar nuestro modelo de **Regresión** en la Semana 3, ahora vamos a auditar la "lógica de precios" de la IA. Utilizaremos **SHAP** para desglosar el valor de las viviendas.

### 💰 El Oráculo Inmobiliario
En este análisis, el impacto se mide en la escala del precio de la vivienda. Queremos confirmar si el modelo ha aprendido reglas de mercado coherentes o si tiene sesgos geográficos.

### 📐 Guía de Lectura Rápida:
* **Orden (Eje Y):** La variable de arriba es la que más influye en el precio final.
* **Sentido (Eje X):** * **Derecha (+):** La característica **sube** el precio de la casa.
    * **Izquierda (-):** La característica **abarata** la propiedad.
* **Color:** 🔴 Rojo (Valor alto de la variable) | 🔵 Azul (Valor bajo).

---

In [ ]:
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# --- 1. CARGA DE RECURSOS ---
# [Especialista]: Cargamos todo desde cero para evitar errores de 'NameError'
path_modelo = '../../../models/modelo_viviendas_optimizado_v2.pkl'
path_datos = '../../../data/processed/viviendas_limpio.csv'

modelo = joblib.load(path_modelo)
df = pd.read_csv(path_datos)

# --- 2. PREPARACIÓN DE DATOS (X) ---
# [Especialista]: Extraemos las columnas exactas que el modelo grabó al entrenar
columnas_entrenamiento = modelo.feature_names_in_

# Reindexamos el DataFrame para que X tenga las mismas 16 columnas y en el mismo orden
X = df.reindex(columns=columnas_entrenamiento, fill_value=0)

# --- 3. CÁLCULO DE VALORES SHAP ---
# [Nota]: Usamos TreeExplainer para modelos de bosque (Random Forest)
print(f"✅ Datos alineados correctamente ({X.shape[1]} columnas).")
print("⏳ Calculando impacto económico... Esto puede tardar unos segundos.")

explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X)

# --- 4. GENERACIÓN DEL GRÁFICO ---
plt.figure(figsize=(12, 8))

# [Ajuste de Versión]: En algunos RandomForest de regresión, SHAP devuelve 
# una matriz 3D o una lista. Forzamos la visualización correcta:
if isinstance(shap_values, list):
    # Si es lista (clases), en regresión solemos querer el primer elemento
    shap.summary_plot(shap_values[0], X, show=False)
elif len(shap_values.shape) == 3:
    # Si es un array 3D (casos, variables, clases), cogemos la primera clase
    shap.summary_plot(shap_values[:, :, 0], X, show=False)
else:
    # Si es un array 2D estándar de regresión
    shap.summary_plot(shap_values, X, show=False)

plt.title("Factores de Valor: ¿Qué influye en el precio en California?", fontsize=16)
plt.xlabel("Impacto SHAP (Derecha: Aumenta Precio | Izquierda: Baja Precio)", fontsize=12)
plt.show()

c:\Users\txema\Documents\IA_Especialista\ia_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Datos alineados correctamente (16 columnas).
⏳ Calculando impacto económico... Esto puede tardar unos segundos.
